# Paridade de engine: masterclass `main` × `release/v0.5` (issue #91)

Mesma história de negócio (trocar `legacy_score` por `score_5`, cutoffs regionais, rating A–E),
rodada **passo a passo** nas duas engines, cada uma no seu venv/worktree, em subprocesso.
Este notebook **não importa `pycreditools`** — só carrega os JSONs que `measure_main.py` e
`measure_v05.py` emitiram (regenere tudo com `run_all.ps1` / `run_all.sh`).

Regras do experimento:

- **Toda métrica sai de `policy.simulate(...).data`** — nenhum fast-path de sweep.
- Contrato de métrica único nos dois lados (ADR 0008, fórmula crua do funil):
  aprovação pré take-up, inad ponderada por contratado.
- Hard filters do challenger **fixados** no conjunto que o sugestor da v0.5 escolheu
  (`vl_negativacao<=0 & vl_vencido_scr<=0 & vl_protestos<=0`) — usado igual nos dois lados.
- **Modo A**: cada engine gera seus dados (`generate_sample_data(seed=7)`).
  **Modo B**: as duas leem o MESMO parquet (`shared_base.parquet`, gerador v0.5 + colunas
  derivadas que a `main` pede). Uma cópia em Excel (`shared_base.xlsx`) fica disponível
  para conferência manual.


In [1]:
import json
from pathlib import Path

import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

R = Path("results")
runs = {name: json.loads((R / f"{name}.json").read_text(encoding="utf-8"))
        for name in ["main_A_60k", "v05_A_60k", "main_B_60k", "v05_B_60k",
                     "main_A_20k", "v05_A_20k"]}
mA, vA = runs["main_A_60k"], runs["v05_A_60k"]
mB, vB = runs["main_B_60k"], runs["v05_B_60k"]

def side_by_side(rows, main_run=None, v05_run=None, digits=4):
    """rows: list of (label, fn(run)). Returns main × v0.5 × diff table."""
    main_run = main_run or mA
    v05_run = v05_run or vA
    out = []
    for label, fn in rows:
        a, b = fn(main_run), fn(v05_run)
        row = {"passo": label, "main": a, "v0.5": b}
        if isinstance(a, (int, float)) and isinstance(b, (int, float)):
            row["diff (v0.5 − main)"] = round(b - a, digits)
        out.append(row)
    return pd.DataFrame(out).set_index("passo")


## Fatos de engine (o que cada branch tem)

Antes dos números: as diferenças de API que os scripts detectaram por introspecção.


In [2]:
facts = ["package_version", "rate_has_observed_col", "optimize_has_directions",
         "has_suggest_hard_filters", "legacy_quantile_exported",
         "calibration_bins_supported", "actual_default_masked_frac"]
pd.DataFrame({"main": {f: mA["engine_facts"][f] for f in facts},
              "v0.5": {f: vA["engine_facts"][f] for f in facts}})


,main,v0.5
package_version,n/a,n/a
rate_has_observed_col,False,True
optimize_has_directions,False,True
has_suggest_hard_filters,False,True
legacy_quantile_exported,False,True
calibration_bins_supported,True,True
actual_default_masked_frac,0.9047,0.9047


## Passo 0 — Os dados são os mesmos? (causa-raiz nº 1)

A issue supunha que os geradores divergem para o mesmo seed. **Medido: não divergem.**
Para `seed=7`, as colunas compartilhadas dos dois geradores são **byte a byte idênticas**
(a v0.5 sorteia `passed_antifraud`/`market_default`/`sample` *depois* do stream comum, então
não desloca o RNG). A prova operacional está abaixo: para cada engine, o modo A (dados
próprios) e o modo B (parquet compartilhado) produzem exatamente as mesmas métricas.

> **Veredito causa nº 1 (dados): descartada.** Todo o desencontro é engine/premissa, não dados.


In [3]:
sections = ["incumbent", "ks_on_hf_approved", "frontier", "champion_iso_approval",
            "three_policies", "regional_cutoffs", "rating", "swap_in_calibration"]
checks = []
for branch, a_run, b_run in [("main", mA, mB), ("v0.5", vA, vB)]:
    row = {"engine": branch}
    for s in sections:
        row[f"{s} (A ≡ B)"] = a_run[s] == b_run[s]
    checks.append(row)
pd.DataFrame(checks).set_index("engine").T


engine,main,v0.5
incumbent (A ≡ B),True,True
ks_on_hf_approved (A ≡ B),True,True
frontier (A ≡ B),True,True
champion_iso_approval (A ≡ B),True,True
three_policies (A ≡ B),True,True
regional_cutoffs (A ≡ B),True,True
rating (A ≡ B),True,True
swap_in_calibration (A ≡ B),True,True


Como A ≡ B, daqui em diante todas as tabelas usam o **modo A a 60k** (o número
“de verdade” que cada lado publicaria) — e ele coincide com o modo B.

## Passo 1 — Incumbente (HF de entrada + cutoff no `legacy_score` q0.78 + take-up)

Única diferença de código: o estágio de take-up.
`main`: `.rate("Take-up", base_rate=1.0, variable="conversion_rate")` (coluna de propensão).
`v0.5`: `.rate("Take-up", base_rate=1.0, observed_col="hired", calibrate_by="score")` (outcome observado).


In [4]:
side_by_side([
    ("aprovação (pré take-up, ADR 0008)", lambda r: r["incumbent"]["approval"]),
    ("inad (ponderada por contratado)",   lambda r: r["incumbent"]["default"]),
    ("contratados",                        lambda r: r["incumbent"]["contracted"]),
    ("take-up",                            lambda r: r["incumbent"]["take_up"]),
    ("inad real (oráculo true_pd)",        lambda r: r["incumbent"]["default_true"]),
    ("aprovação pós take-up (contrato antigo)",
        lambda r: r["incumbent"]["legacy_contract"]["approval_post_take_up"]),
    ("inad não ponderada (contrato antigo)",
        lambda r: r["incumbent"]["legacy_contract"]["default_unweighted"]),
])


,main,v0.5,diff (v0.5 − main)
passo,,,
"aprovação (pré take-up, ADR 0008)",0.2055,0.2055,0.0000
inad (ponderada por contratado),0.0754,0.0754,0.0000
contratados,"5,719.0000","5,719.0000",0.0000
take-up,0.4639,0.4639,0.0000
inad real (oráculo true_pd),0.0707,0.0707,0.0000
aprovação pós take-up (contrato antigo),0.0953,0.0953,0.0000
inad não ponderada (contrato antigo),0.0754,0.0754,0.0000


**Leitura.** Idênticos até o 4º decimal, inclusive contratados (5.719). No incumbente não
há swap-in (a política reproduz a aprovação vigente), então os dois idiomas de take-up
convergem para o volume observado — e batem com a referência da issue (20,5% / 7,5% / 5.719).

Também recomputamos o **contrato antigo** de métrica nos dois lados: mesmo número nos dois.
> **Veredito causa nº 2 (contrato de métrica): não gera gap numérico** quando a mesma fórmula
> é aplicada aos mesmos dados. O ADR 0008 muda *rótulo e denominador reportados*, não o funil.
> Diferenças de manchete README × masterclass vêm de qual fórmula cada texto escolheu exibir.

## Passo 2 — Hard filters do challenger (causa-raiz nº 5)

Na v0.5 rodamos `suggest_hard_filters` (bad_col=`actual_default`, floor 0,55, lift ≥ 1,3);
na `main` o sugeridor não existe. O conjunto sugerido foi **exatamente** o conjunto que
fixamos nos dois lados — ou seja, a seleção de HF não pode explicar diferença nenhuma aqui.


In [5]:
hf_v, hf_m = vA["hard_filters"], mA["hard_filters"]
print("regras fixadas (ambas engines):", hf_m["used_rules"])
print("taxa de passagem dos HF:        ", f"{hf_m['hf_pass_rate']:.4f} (idêntica nos dois lados)")
print()
print("sugestor v0.5 escolheu:", hf_v["suggested"]["rules"])
print("orçamento do sugestor: ", {k: round(v, 3) for k, v in hf_v["suggested"]["budget"].items()})


regras fixadas (ambas engines): ['vl_negativacao lte 0', 'vl_vencido_scr lte 0', 'vl_protestos lte 0']
taxa de passagem dos HF:         0.5769 (idêntica nos dois lados)

sugestor v0.5 escolheu: ['vl_negativacao lte 0', 'vl_vencido_scr lte 0', 'vl_protestos lte 0']
orçamento do sugestor:  {'floor': 0.55, 'spent': 0.423, 'approval_rate': 0.577, 'headroom': 0.027}


> **Veredito causa nº 5 (seleção de HF): descartada por construção** — e o sugestor,
> deixado livre, escolhe o mesmo conjunto que o README usa à mão.

## Passo 3 — KS por score, só sobre os aprovados pelos HF

`ModelEvaluator.compute_ks` sobre `actual_default`, subpopulação `HF == True`, nos dois lados.


In [6]:
ks = pd.DataFrame({"main": mA["ks_on_hf_approved"], "v0.5": vA["ks_on_hf_approved"]})
ks["diff"] = ks["v0.5"] - ks["main"]
ks.sort_values("v0.5", ascending=False)


,main,v0.5,diff
score_5,0.3333,0.3333,0.0000
score_4,0.3196,0.3196,0.0000
score_3,0.2975,0.2975,0.0000
score_2,0.2947,0.2947,0.0000
legacy_score,0.2757,0.2757,0.0000


**Leitura.** KS idêntico (score_5 = 0,3333, batendo a referência da issue). Ranking igual;
`score_5` vence nos dois lados. Nenhuma divergência de avaliação de modelo.

## Passos 4–5 — Challenger `score_5`: fronteira e política iso-aprovação

Grade de cutoffs idêntica (quantis do `score_5`), cada ponto simulado com `.simulate()`.
O parâmetro `calibration_bins` **existe nas duas engines**; a diferença é de *fluxo publicado*:
a masterclass da v0.5 seta `calibration_bins=5` (resposta recomendada ao
`CalibrationReliabilityWarning`, que o PR #72 introduziu), enquanto o fluxo do README/main
fica nos decis default. Cutoff iso-aprovação: os dois lados escolhem **736**.


In [7]:
ch = lambda r: r["champion_iso_approval"]
side_by_side([
    ("cutoff escolhido",                     lambda r: ch(r)["cutoff"]),
    ("aprovação",                             lambda r: ch(r)["approval"]),
    ("contratados",                           lambda r: ch(r)["contracted"]),
    ("inad simulada (sem stress)",            lambda r: ch(r)["default"]),
    ("inad simulada (stress ×1,3)",           lambda r: ch(r)["default_stress_1.3"]),
    ("inad simulada (stress ×1,5)",           lambda r: ch(r)["default_stress_1.5"]),
    ("inad real (oráculo true_pd)",           lambda r: ch(r)["default_true"]),
    ("inad do incumbente (referência)",       lambda r: r["incumbent"]["default"]),
])


,main,v0.5,diff (v0.5 − main)
passo,,,
cutoff escolhido,736.0000,736.0000,0.0000
aprovação,0.1992,0.1992,0.0000
contratados,"5,526.0500","5,573.6907",47.6407
inad simulada (sem stress),0.0584,0.0568,-0.0016
"inad simulada (stress ×1,3)",0.0664,0.0644,-0.0020
"inad simulada (stress ×1,5)",0.0718,0.0695,-0.0023
inad real (oráculo true_pd),0.0639,0.0644,0.0005
inad do incumbente (referência),0.0754,0.0754,0.0000


**Leitura, com carinho:**

- **O ganho é real.** Contra o oráculo (`true_pd`), na mesma aprovação (~19,9%) o challenger
  entrega inad **6,4%** contra **7,5%** do incumbente, nas *duas* engines (−1,1 p.p.).
- **Sem stress, as duas engines exageram o ganho** (5,84% na main, 5,68% na v0.5 — ambas
  abaixo do real 6,4%), porque a calibração por keep-in subestima o PD do swap-in.
- **Com stress ×1,5 o ganho quase some** (7,18% / 6,95% vs 7,5% do incumbente) — é o
  “modelo melhor sem ganho” da issue. O dial abaixo mostra que ×1,5 **passa do ponto**.
- A diferença v0.5 − main na inad simulada (−0,16 p.p.) vem de duas fontes, decompostas
  no quadro de quadrantes: `calibration_bins=5` (causa nº 4) e o idioma de take-up (causa nº 3).

### Quadrantes no cutoff 736


In [8]:
quads = []
for scen in ["keep_in", "swap_in", "swap_out", "keep_out"]:
    qm = ch(mA)["quadrants"].get(scen, {})
    qv = ch(vA)["quadrants"].get(scen, {})
    quads.append({
        "quadrante": scen,
        "n (main)": qm.get("n"), "n (v0.5)": qv.get("n"),
        "vol contratado (main)": qm.get("vol"), "vol contratado (v0.5)": qv.get("vol"),
        "PD sim (main)": qm.get("sim"), "PD sim (v0.5)": qv.get("sim"),
        "PD real (main)": qm.get("true"), "PD real (v0.5)": qv.get("true"),
        "inad observada (main)": qm.get("actual"), "inad observada (v0.5)": qv.get("actual"),
    })
pd.DataFrame(quads).set_index("quadrante")


,n (main),n (v0.5),vol contratado (main),vol contratado (v0.5),PD sim (main),PD sim (v0.5),PD real (main),PD real (v0.5),inad observada (main),inad observada (v0.5)
quadrante,,,,,,,,,,
keep_in,8301,8301,"3,715.0000","3,715.0000",0.0471,0.0471,0.0460,0.0460,0.0471,0.0471
swap_in,3653,3653,"1,811.0500","1,858.6907",0.0816,0.0762,0.1005,0.1010,NaN,NaN
swap_out,4027,4027,0.0000,0.0000,NaN,NaN,NaN,NaN,0.1277,0.1277
keep_out,44019,44019,0.0000,0.0000,NaN,NaN,NaN,NaN,NaN,NaN


**Leitura.**

- As **populações são idênticas** (mesmos n por quadrante — consequência do passo 0).
- **Causa nº 3 (take-up)**: o volume contratado de swap-in difere — 1.811 (main, coluna de
  propensão `conversion_rate`) vs 1.859 (v0.5, `observed_col="hired"` calibrado por score),
  +2,6%. Efeito pequeno mas real no denominador da inad: 5.526 vs 5.574 contratados no total.
- **Causa nº 4 (PD do swap-in)**: PD simulado do swap-in 8,16% (main) vs 7,62% (v0.5-bins5),
  contra **10,1% real**. As duas subestimam; a v0.5 com bins=5 subestima *mais* — ver dial.
- swap-out idêntico (12,77% observado): quem o challenger corta é igual nos dois lados.

### O dial do swap-in PD (causa-raiz nº 4, o coração do desencontro)


In [9]:
sc_m, sc_v = mA["swap_in_calibration"], vA["swap_in_calibration"]
ch_m, ch_v = ch(mA), ch(vA)
dial = pd.DataFrame([
    {"medida": "imputado (calibração default, decis)", "main": sc_m["swap_in_pd_imputed"],
     "v0.5": sc_v.get("swap_in_pd_imputed_naive")},
    {"medida": "imputado (calibration_bins=5)", "main": sc_m.get("swap_in_pd_imputed_bins5"),
     "v0.5": sc_v["swap_in_pd_imputed"]},
    {"medida": "imputado × stress 1,3 (derivado)", "main": sc_m["swap_in_pd_stress_1.3"],
     "v0.5": sc_v["swap_in_pd_stress_1.3"]},
    {"medida": "imputado × stress 1,5 (derivado)", "main": sc_m["swap_in_pd_stress_1.5"],
     "v0.5": sc_v["swap_in_pd_stress_1.5"]},
    {"medida": "stress 1,5 MEDIDO via policy.stress(1.5)",
     "main": ch_m["swap_in_pd_stress_1.5_measured"],
     "v0.5": ch_v["swap_in_pd_stress_1.5_measured"]},
    {"medida": "REAL (oráculo true_pd)", "main": sc_m["swap_in_pd_true"],
     "v0.5": sc_v["swap_in_pd_true"]},
]).set_index("medida")
dial


,main,v0.5
medida,,
"imputado (calibração default, decis)",0.0816,0.0813
imputado (calibration_bins=5),0.0760,0.0762
"imputado × stress 1,3 (derivado)",0.1060,0.0991
"imputado × stress 1,5 (derivado)",0.1224,0.1143
"stress 1,5 MEDIDO via policy.stress(1.5)",0.1224,0.1143
REAL (oráculo true_pd),0.1005,0.1010


**Leitura.**

- Com a **mesma calibração** (decis default), as engines quase coincidem: 8,16% vs 8,13%.
  O resíduo (0,03 p.p.) é só o peso de take-up. E setando `calibration_bins=5` **na main
  também** (o parâmetro existe nas duas engines), os imputados voltam a se colar. Ou seja:
  **não há bug de calibração entre engines** — há uma *escolha de fluxo* diferente
  (masterclass seta bins=5; README/main fica nos decis default).
- Contra o real (10,1%), o imputado cru subestima ~20–25%. **Markup honesto ≈ ×1,3**:
  8,16 × 1,3 = 10,6% e 7,62 × 1,3 = 9,9%, cercando o real por cima e por baixo.
- **Stress ×1,5 superestima** (11,4–12,2% vs 10,1% real). É exatamente o mecanismo que
  fazia o “modelo melhor” parecer sem ganho: o markup exagerado devolve ao challenger
  o risco que ele de fato não tem.
- A linha “stress 1,5 MEDIDO” veio de um `.simulate()` com `policy.stress(1.5)` real em cada
  engine — coincide com a derivação analítica, validando que as duas implementam
  `AggravationStress` como `clip(pd × fator)` no caminho de código de verdade.

> **Confirmado o critério de aceite da issue:** o “modelo melhor sem ganho” era artefato do
> stress ×1,5. Contra `true_pd` o ganho reaparece (−1,1 p.p.), e com markup ~×1,3 a inad
> simulada fica colada no real.

### Três políticas na fronteira (localizadas pela inad simulada sem stress)


In [10]:
pols = []
for pm, pv in zip(mA["three_policies"], vA["three_policies"]):
    pols.append({
        "política": pm["name"],
        "cutoff (main)": pm["cutoff"], "cutoff (v0.5)": pv["cutoff"],
        "aprovação (main)": pm["approval"], "aprovação (v0.5)": pv["approval"],
        "inad sim (main)": pm["default"], "inad sim (v0.5)": pv["default"],
        "inad real (main)": pm["default_true"], "inad real (v0.5)": pv["default_true"],
        "contratados (main)": pm["contracted"], "contratados (v0.5)": pv["contracted"],
    })
pd.DataFrame(pols).set_index("política")


,cutoff (main),cutoff (v0.5),aprovação (main),aprovação (v0.5),inad sim (main),inad sim (v0.5),inad real (main),inad real (v0.5),contratados (main),contratados (v0.5)
política,,,,,,,,,,
iso_approval,736,736,0.1992,0.1992,0.0584,0.0568,0.0639,0.0644,"5,526.0500","5,573.6907"
iso_default,686,661,0.2323,0.2486,0.0700,0.0741,0.0786,0.0862,"6,653.1300","7,247.7513"
balanced,710,710,0.2157,0.2157,0.0662,0.0633,0.0709,0.0716,"6,064.4300","6,131.8398"


**Leitura.**

- **iso-aprovação**: mesmo cutoff (736) nos dois lados — a fronteira de aprovação é idêntica
  porque as populações são idênticas.
- **iso-inad**: aqui as engines divergem de verdade — main corta em 686 (aprovação 23,2%),
  v0.5 em 661 (24,9%). Não é bug: a v0.5, imputando PD menor no swap-in (bins=5), *acha*
  que cabe descer mais o corte. É a causa nº 4 propagada para a decisão.
- **Custo da imputação otimista, nos dois lados**: a política “iso-inad” escolhida pela inad
  simulada entrega inad **real** acima do alvo (main 7,9%, v0.5 8,6%, alvo 7,54%). Lição
  operacional: **localizar cortes com markup ~×1,3**, não com o imputado cru — em qualquer
  uma das engines.

## Passo 6 — Cutoffs regionais vs corte geral (causa-raiz nº 6)

Alvo comum: a inad do incumbente (7,54%). Corte geral e cortes por região escolhidos pela
inad simulada (sem stress) de cada engine — mesma grade, mesmo critério.


In [11]:
reg_rows = []
reg_m = {r["region"]: r for r in mA["regional_cutoffs"]["regions"]}
reg_v = {r["region"]: r for r in vA["regional_cutoffs"]["regions"]}
for region in sorted(reg_m):
    a, b = reg_m[region], reg_v[region]
    reg_rows.append({"região": region, "n": a["applicants"],
                     "cutoff (main)": a["cutoff"], "cutoff (v0.5)": b["cutoff"],
                     "aprovação (main)": a["approval"], "aprovação (v0.5)": b["approval"],
                     "inad sim (main)": a["default"], "inad sim (v0.5)": b["default"]})
display(pd.DataFrame(reg_rows).set_index("região"))

tot = []
for name, key in [("corte geral", "general"), ("segmentado (soma das regiões)", "segmented_total")]:
    gm, gv = mA["regional_cutoffs"][key], vA["regional_cutoffs"][key]
    tot.append({"política": name,
                "aprovação (main)": gm["approval"], "aprovação (v0.5)": gv["approval"],
                "inad sim (main)": gm["default"], "inad sim (v0.5)": gv["default"],
                "contratados (main)": gm["contracted"], "contratados (v0.5)": gv["contracted"]})
pd.DataFrame(tot).set_index("política")


,n,cutoff (main),cutoff (v0.5),aprovação (main),aprovação (v0.5),inad sim (main),inad sim (v0.5)
região,,,,,,,
Centro-Oeste,5970,752,752,0.1811,0.1811,0.0616,0.0601
Nordeste,10812,760,735,0.1665,0.1845,0.0734,0.0703
Norte,4125,704,676,0.2007,0.2165,0.0663,0.0749
Sudeste,26989,669,643,0.2475,0.2635,0.0669,0.0706
Sul,12104,680,575,0.2500,0.3141,0.0604,0.0750


,aprovação (main),aprovação (v0.5),inad sim (main),inad sim (v0.5),contratados (main),contratados (v0.5)
política,,,,,,
corte geral,0.2323,0.2486,0.0700,0.0741,"6,653.1300","7,247.7513"
segmentado (soma das regiões),0.2236,0.2480,0.0659,0.0712,"6,373.0800","7,204.1389"


**Leitura.**

- Os cutoffs regionais divergem (ex.: Sul 680 na main vs 575 na v0.5) pelo **mesmo e único
  motivo** do passo 5: a inad simulada da v0.5 é mais otimista (bins=5 + take-up), então ela
  desce mais o corte até “gastar” o mesmo orçamento de risco. Downstream da causa nº 4,
  não uma causa independente.
- O “5,80% de PD estressado” do README não é reproduzível por nenhuma das engines com o fluxo
  canônico: era um alvo escolhido à mão com outra premissa de stress. Diferença de *setup*,
  não de engine.

### Bônus do passo 6 — o `optimize_cutoffs` de cada engine, checado contra `.simulate()`

A comparação acima usou a mesma grade de `.simulate()` nos dois lados (regra de ouro).
Aqui deixamos **cada engine usar seu próprio otimizador** (como a masterclass e o README
fazem) com o mesmo alvo, e re-reportamos as métricas do corte escolhido via `.simulate()`.


In [12]:
oc = []
for tag, r in [("main", mA), ("v0.5", vA)]:
    o = r["optimizer_check"]
    oc.append({"engine": tag, "cutoff escolhido": o["cutoff"],
               "inad alegada pelo otimizador": o["optimizer_metrics"]["overall_default_rate"],
               "inad re-simulada (.simulate)": o["resimulated"]["default"],
               "inad real (oráculo)": o["resimulated"]["default_true"],
               "aprovação": o["resimulated"]["approval"],
               "alvo": r["regional_cutoffs"]["target_default"]})
pd.DataFrame(oc).set_index("engine")


,cutoff escolhido,inad alegada pelo otimizador,inad re-simulada (.simulate),inad real (oráculo),aprovação,alvo
engine,,,,,,
main,625,0.0680,0.0848,0.0975,0.2711,0.0754
v0.5,688,0.0740,0.0669,0.0780,0.2306,0.0754


**Leitura — o achado mais operacional do estudo.**

- O otimizador da **main** alega inad 6,8% no corte 625, mas o `.simulate()` no mesmo corte
  dá **8,5%** — fura o alvo de 7,54% em ~1 p.p. O fast-path da main erra pro lado
  **otimista** (subestima risco): quem calibrou cutoff regional pelo otimizador da main
  comprou mais risco do que pensava.
- O otimizador da **v0.5** alega 7,4% no corte 688 e o `.simulate()` dá **6,7%** — erra pro
  lado **conservador** (é a inflação do swap-in no fast-path que a issue #91 já tinha
  flagrado: 0,086 vs 0,076). Deixa aprovação na mesa, mas não fura alvo.
- Nos dois casos a regra de ouro se confirma: **fast-path localiza, `.simulate()` reporta.**

## Passo 7 — Rating A–E e validação DEV/OOT

`fit_risk_groups(score_5, actual_default, bins=20, max_groups=5, min_vol_ratio=0.05,
max_crossings=3, time_col="safra", oot_date="2025-01")` sobre os contratados, nos dois lados.


In [13]:
rt = []
rm = {(r["grade"], r["period"]): r["pd"] for r in mA["rating"]["rows"]}
rv = {(r["grade"], r["period"]): r["pd"] for r in vA["rating"]["rows"]}
for grade in "ABCDE":
    rt.append({"grade": grade,
               "PD Train (main)": rm.get((grade, "Train")), "PD Train (v0.5)": rv.get((grade, "Train")),
               "PD OOT (main)": rm.get((grade, "OOT")), "PD OOT (v0.5)": rv.get((grade, "OOT"))})
pd.DataFrame(rt).set_index("grade")


,PD Train (main),PD Train (v0.5),PD OOT (main),PD OOT (v0.5)
grade,,,,
A,0.0138,0.0138,0.0239,0.0239
B,0.0525,0.0525,0.0807,0.0807
C,0.0781,0.0781,0.0782,0.0782
D,0.1240,0.1240,0.1123,0.1123
E,0.1757,0.1757,0.1769,0.1769


**Leitura.** Grades e PDs **idênticos** (5 grupos, amplitude Train 1,4% → 17,6%).
O motor de rating não mudou entre as branches.

## Sensibilidade a tamanho: n = 20.000 (default da `main`)


In [14]:
m20, v20 = runs["main_A_20k"], runs["v05_A_20k"]
sens = []
for tag, r60, r20 in [("main", mA, m20), ("v0.5", vA, v20)]:
    for n_lbl, r in [("60k", r60), ("20k", r20)]:
        c = r["champion_iso_approval"]
        sens.append({"engine": tag, "n": n_lbl,
                     "inad incumbente": r["incumbent"]["default"],
                     "challenger sim": c["default"],
                     "challenger real (oráculo)": c["default_true"],
                     "challenger stress ×1,5": c["default_stress_1.5"]})
pd.DataFrame(sens).set_index(["engine", "n"])


inad incumbente  challenger sim  challenger real (oráculo)  challenger stress ×1,5
engine n                                                                                      
main   60k           0.0754          0.0584                     0.0639                  0.0718
       20k           0.0712          0.0540                     0.0695                  0.0670
v0.5   60k           0.0754          0.0568                     0.0644                  0.0695
       20k           0.0712          0.0529                     0.0693                  0.0652

**Leitura.** Em 20k o veredito não muda: o challenger continua melhor que o incumbente
contra o oráculo nas duas engines, com margem menor (6,9% vs 7,1%) — coerente com uma base
3× menor. Nenhuma conclusão deste estudo depende do tamanho da amostra.

## Warnings capturados (`notes`)

A v0.5 emite `CalibrationReliabilityWarning` (PR #72) durante a varredura — capturado, não
suprimido. A `main` emite só o aviso genérico de decis.


In [15]:
import textwrap
for tag, r in [("main", mA), ("v0.5", vA)]:
    uniq = []
    for n_ in r["notes"]:
        key = n_[:80]
        if key not in [u[:80] for u in uniq]:
            uniq.append(n_)
    print(f"— {tag}: {len(r['notes'])} warnings, {len(uniq)} distintos")
    for u in uniq:
        print(textwrap.fill(u, 110, initial_indent="   • ", subsequent_indent="     "))
    print()


— main: 1 warnings, 1 distintos
   • [UserWarning] Swap-in PD imputation is using the default of 10 score bins (deciles). Pass
     CreditPolicy(calibration_bins=...) to set a different granularity.

— v0.5: 28 warnings, 28 distintos
   • [UserWarning] 'actual_default' is observed for 9.5% of the base; every lift is measured among the
     contracted, a population the incumbent policy already selected. See the measurement caveat in
     suggest_hard_filters' docstring.
   • [CalibrationReliabilityWarning] Swap-in PD calibration is unreliable: 10% of swap-ins score outside the
     keep-ins' observed range, so their imputed PD is edge-clamp extrapolation, not measurement. No bin count
     fixes this — the score range has no observed defaults to measure. Pass an estimated_default_col with a
     model PD.
   • [CalibrationReliabilityWarning] Swap-in PD calibration is unreliable: 6% of swap-ins score outside the
     keep-ins' observed range, so their imputed PD is edge-clamp extrapolati

## Veredito por causa-raiz

| # | Causa candidata | Veredito | Evidência |
|---|---|---|---|
| 1 | Dados diferentes pro mesmo seed | **Descartada** | Colunas compartilhadas byte a byte idênticas; modo A ≡ modo B nas duas engines (passo 0) |
| 2 | Contrato de métrica (ADR 0008) | **Sem gap numérico** | Mesma fórmula nos mesmos dados → mesmos números; contrato antigo recomputado também coincide (passo 1) |
| 3 | Take-up (propensão × observed_col) | **Esperada, pequena** | +2,6% de volume swap-in na v0.5 (1.859 vs 1.811); mecanismo do ADR 0008 / RateStage genérico (#68) |
| 4 | Swap-in PD + stress | **Esperada, é O driver** | Decis ≈ decis (8,16% vs 8,13%); `calibration_bins=5` (PR #72) desloca para 7,62%; real 10,1%; ×1,3 honesto, ×1,5 exagera |
| 5 | Seleção de HF | **Descartada** | Sugestor (#73) escolhe exatamente o conjunto fixado; HF pass rate idêntica |
| 6 | Cutoffs regionais | **Downstream da nº 4** | Cortes divergem só porque a inad simulada diverge; critério e grade idênticos |

Detalhes e recomendação de merge: `validation/README.md`.
